# 1 · Estrazione RADIOMICA (pyradiomics)

Estrae le feature radiomiche (forma + intensità + texture) dalla ROI **pancreas + lesione**
di ogni TC, e le salva in `cache_features/feat_radiomics.parquet`.

> ⚠️ **Kernel: "PANORAMA Radiomica (.venv 3.7)"** — pyradiomics ha wheel solo per Python 3.5–3.7.
> ♻️ **Resumabile**: se interrompi, ri-eseguendo continua dai casi mancanti.
> Tutta la logica è **qui dentro** (nessun file esterno).

In [1]:
# --- Setup ---

# --- Setup, percorsi e RIPRODUCIBILITÀ --------------------------------------
import os, warnings, random
import numpy as np, pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED); random.seed(SEED); np.random.seed(SEED)

# trova la radice del progetto (contiene imagesTr/ e il file clinico)
CWD = Path.cwd()
def _e_radice(p): return (p / "imagesTr").exists() and (p / "cache" / "clinical_information.xlsx").exists()
PROJECT_DIR = next((p for p in [CWD, *CWD.parents] if _e_radice(p)), CWD)
IMAGES_DIR = PROJECT_DIR / "imagesTr"
LABELS_DIR = PROJECT_DIR / "labelsTr"
CLINICAL   = PROJECT_DIR / "cache" / "clinical_information.xlsx"
CACHE_FEAT = PROJECT_DIR / "ml_study" / "cache_features"; CACHE_FEAT.mkdir(parents=True, exist_ok=True)

# Costanti delle maschere PANORAMA: 0=sfondo,1=lesione,2=vene,3=arterie,4=pancreas,5=dotto,6=coledoco
LESION_CLASS, PANCREAS_CLASS = 1, 4
print("Radice progetto:", PROJECT_DIR, "| immagini:", IMAGES_DIR.exists(), "| clinico:", CLINICAL.exists())

import SimpleITK as sitk
from radiomics import featureextractor, imageoperations
print("pyradiomics pronto.")

Radice progetto: D:\PANORAMA_v2 | immagini: True | clinico: True
pyradiomics pronto.


## 1) La coorte (326 casi)
Teniamo i casi **scaricati** con **sesso ed età** presenti (esclude i sottoinsiemi pubblici
NIH/MSD privi di quei metadati → restano i centri olandesi RUMC + UMCG).
- `y` = 1 se PDAC, 0 se controllo · `sex_bin` = 0 F / 1 M · `age` in anni (da `'077Y'` a 77).

In [2]:
df = pd.read_excel(CLINICAL).rename(columns={"PANORAMA_study_id": "study_id"})
df["age"] = pd.to_numeric(df["patient_age"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
df["sex_bin"] = df["patient_sex"].astype(str).str.strip().str.upper().str[0].map({"F": 0, "M": 1})
df["y"] = (df["label"].astype(str).str.upper() == "PDAC").astype(int)
df["ha_img"] = df["study_id"].map(lambda s: (IMAGES_DIR / f"{s}_0000.nii.gz").exists())
cohort = df[df["ha_img"] & df["sex_bin"].notna() & df["age"].notna()][
    ["study_id", "y", "age", "sex_bin", "label", "level"]].reset_index(drop=True)
print("Casi nella coorte:", len(cohort))
print(cohort["y"].map({1: "PDAC", 0: "controllo"}).value_counts().to_string())
cohort.head()

Casi nella coorte: 326
PDAC         182
controllo    144


,study_id,y,age,sex_bin,label,level
0,100012_00001,0,73.0,1.0,non-PDAC,radiology
1,100030_00001,1,51.0,0.0,PDAC,histopathology
2,100031_00001,0,38.0,1.0,non-PDAC,radiology
3,100037_00001,0,23.0,1.0,non-PDAC,radiology
4,100043_00001,1,73.0,0.0,PDAC,pathology


## 2) I parametri di pyradiomics (spiegati)
La ROI è la **regione della maschera** (pancreas classe 4 + lesione classe 1), non un box.

In [3]:
# ┌── PARAMETRI DI ESTRAZIONE pyradiomics ───────────────────────────────────────┐
# │ binCount = 50           50 bin a conteggio fisso per discretizzare le intensità│
# │ normalize = True        normalizza intensità (media0/std1) -> riduce diff. scanner│
# │ removeOutliers = 3      scarta voxel oltre ±3σ (artefatti/metallo/aria)         │
# │ interpolator = BSpline  interpolazione liscia per il ricampionamento           │
# │ resampledPixelSpacing = [0.8, 0.8, 0]                                          │
# │     0.8×0.8 mm nel PIANO (≈ mediana nativa in-plane 0.39–0.98) -> feature       │
# │     confrontabili tra scanner con upsampling minimo. z=0 -> NON ricampiona lungo│
# │     z (lo spessore slice PANORAMA è 0.45–5.0 mm, troppo eterogeneo per il 3D).  │
# │ force2D = True          texture calcolata FETTA-PER-FETTA (2D), coerente col z. │
# │ solo immagine 'Original' (LoG e Wavelet disabilitati)                          │
# └────────────────────────────────────────────────────────────────────────────────┘
settings = {
    "binCount": 50, "normalize": True, "removeOutliers": 3,
    "interpolator": sitk.sitkBSpline,
    "resampledPixelSpacing": [0.8, 0.8, 0],   # se [1,1,0] -> errore "single voxel"
    "force2D": True,
    "correctMask": True, "geometryTolerance": 1e-3,
}
extractor = featureextractor.RadiomicsFeatureExtractor(**settings)
extractor.settings.pop("binWidth", None)     # usiamo binCount, non binWidth
extractor.disableAllImageTypes(); extractor.enableImageTypeByName("Original")

def maschera_pancreas_lesione(mask_file):
    """ROI binaria = 1 dove label è pancreas(4) o lesione(1)."""
    m = sitk.ReadImage(str(mask_file)); arr = sitk.GetArrayFromImage(m)
    b = ((arr == LESION_CLASS) | (arr == PANCREAS_CLASS)).astype(np.uint8)
    if b.sum() == 0: return None
    bm = sitk.GetImageFromArray(b); bm.CopyInformation(m); return bm
print("Estrattore configurato.")

Fixed bin Count enabled! However, we recommend using a fixed bin Width. See http://pyradiomics.readthedocs.io/en/latest/faq.html#radiomics-fixed-bin-width for more details


Estrattore configurato.


## 3) Estrazione (resumabile) e salvataggio
Per ogni caso: carica immagine + maschera → binarizza (pancreas+lesione) → `checkMask` corregge
la geometria → estrae le feature `original_*`. Salva ogni 10 casi in `feat_radiomics.parquet`.

In [4]:
out_path = CACHE_FEAT / "feat_radiomics.parquet"
prev = pd.read_parquet(out_path) if out_path.exists() else None
done = set(prev["study_id"]) if prev is not None else set()
todo = [s for s in cohort["study_id"] if s not in done]
print(f"Già in cache: {len(done)} | da estrarre: {len(todo)}")

rows, falliti = [], []
for i, sid in enumerate(todo, 1):
    try:
        img = sitk.ReadImage(str(IMAGES_DIR / f"{sid}_0000.nii.gz"))
        bm = maschera_pancreas_lesione(LABELS_DIR / f"{sid}.nii.gz")
        if bm is None: raise RuntimeError("ROI vuota")
        _, corr = imageoperations.checkMask(img, bm)
        if corr is not None: bm = corr
        res = extractor.execute(img, bm, label=1)
        rec = {("rad_" + k[len("original_"):]): float(v) for k, v in res.items() if k.startswith("original_")}
    except Exception as e:
        print(f"  ⚠️ {sid}: fallito ({type(e).__name__}) -> NaN"); falliti.append(sid); rec = {}
    rec["study_id"] = sid; rows.append(rec)
    if i % 10 == 0 or i == len(todo):
        print(f"  {i}/{len(todo)}")
        cur = pd.DataFrame(rows)
        (pd.concat([prev, cur], ignore_index=True) if prev is not None else cur).to_parquet(out_path, index=False)
print(f"\n✅ FATTO. Falliti: {len(falliti)}. Cache: {out_path}")
rad = pd.read_parquet(out_path); print(f"Radiomica: {rad.shape[1]-1} feature × {len(rad)} casi")

Già in cache: 326 | da estrarre: 0

✅ FATTO. Falliti: 0. Cache: D:\PANORAMA_v2\ml_study\cache_features\feat_radiomics.parquet
Radiomica: 107 feature × 326 casi
